# Estimate Hallucination Rates Across Multiple Regions with Stratified PPI++

**Stratified PPI++** — extends PPI++ to datasets naturally partitioned into strata (e.g. by region, language, or data source). It adapts the bias-correction per stratum, yielding narrower confidence intervals when strata differ in proxy quality or hallucination rates. This guide walks through a realistic scenario where your global product handles conversations from different regions.

---

**What you will learn:**

- Why LLM-as-Judge metrics are **regionally biased**
- How Stratified PPI++ adapts to different strata simultaneously
- How per-stratum lambda tuning narrows confidence intervals
- Why ignoring strata leaves precision on the table

## The problem: your LLM judge has different biases in different regions

Let's assume you run a global customer-facing assistant handling conversations in three regions: **North America**, **Europe**, and **Asia**.

### The signals

You deploy an LLM judge to measure hallucination rates per region. However, you notice something troubling:

| Region | Judge Reports | User Complaints | Discrepancy |
|--------|---------------|-----------------|-------------|
| **North America** | 5% | 8% | Judge too optimistic by 3 pp |
| **Europe** | 7% | 10% | Judge too optimistic by 3 pp |
| **Asia** | 8% | 15% | Judge too optimistic by 7 pp |

The bias is **regional**: the judge systematically underestimates hallucinations, but the severity varies. Asia (likely due to language and translation nuances) sees the worst judge performance.

### The challenge

You have:
- **~4,000 LLM judgements** (cheap, fast, but regionally biased)
- **~300 human annotations** spread across three regions (accurate, but sparse)

**Stratified PPI++ combines both** to produce a reliable, unbiased global estimate while respecting regional differences in proxy quality.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

from glide.core.simulated_datasets import generate_stratified_binary_dataset
from glide.estimators import ClassicalMeanEstimator, PPIMeanEstimator, StratifiedPPIMeanEstimator
from glide.io.export import to_json

# ── Colour palette ──────────────────────────────────────────
C_JUDGE = "#E74C3C"  # LLM judge  — red-orange
C_HUMAN = "#2E86AB"  # Human-only — blue
C_GLIDE = "#27AE60"  # GLIDE      — green
C_TRUTH = "#2C3E50"  # True value — dark slate
C_NA = "#3498DB"  # North America — lighter blue
C_EU = "#E67E22"  # Europe — orange
C_ASIA = "#E74C3C"  # Asia — red

# ── Global plot style ───────────────────────────────────────
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "#FAFAFA",
        "axes.grid": True,
        "grid.color": "#E5E5E5",
        "grid.linewidth": 0.8,
        "axes.titlesize": 14,
        "axes.titlepad": 14,
        "axes.labelsize": 12,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
    }
)

## Simulating thousands of conversations across three regions

`generate_stratified_binary_dataset` produces synthetic data mirroring the scenario above: three regions with different hallucination rates and different LLM judge biases.

Each **record** represents one conversation, tagged with its region:

| Field | Present for | Meaning |
|-------|-------------|----------|
| `y_true` | Labeled records only | `1` = hallucination confirmed by human |
| `y_proxy` | All records | `1` = hallucination flagged by LLM judge |
| `stratum_id` | All records | `0` = North America, `1` = Europe, `2` = Asia |

In [ ]:
# Generate stratified data: 3 regions with different hallucination rates and judge biases
labeled, unlabeled = generate_stratified_binary_dataset(
    n=[100, 100, 100],  # ~100 annotations per region
    N=[1000, 1500, 1500],  # ~1000-1500 judge verdicts per region
    true_mean=[0.08, 0.10, 0.15],  # True hallucination rates: NA=8%, EU=10%, Asia=15%
    proxy_mean=[0.05, 0.07, 0.08],  # Judge reports: NA=5%, EU=7%, Asia=8%
    correlation=[0.70, 0.65, 0.55],  # Judge quality: worse in Asia
    random_seed=42,
)

dataset = labeled + unlabeled

# Label strata for readability
region_names = {0: "North America", 1: "Europe", 2: "Asia"}
for record in dataset:
    record["region"] = region_names[record["stratum_id"]]

print(f"Total conversations : {len(dataset):,}")
print(f"  Manually annotated : {len(labeled):,}")
print(f"  LLM-judged only    : {len(unlabeled):,}")
print()
print("Distribution by region:")
for region_id, region_name in region_names.items():
    count = sum(1 for r in dataset if r.get("stratum_id") == region_id)
    labeled_count = sum(1 for r in labeled if r.get("stratum_id") == region_id)
    print(f"  {region_name:<15} : {count:,} total ({labeled_count} annotated)")

## Naive approaches fail: ignore regions or pool data

Three obvious approaches each have a fatal flaw:

**Option A — Trust the judge on all conversations (ignoring regions).**  
Precise (large sample), but the judge's regional biases corrupt the estimate.

**Option B — Trust only human annotations (ignoring regions).**  
Unbiased, but the 95% confidence interval is very wide.

**Option C — Estimate per region separately, then average.**  
Respects regional structure, but each region gets few labeled examples, leading to wide per-region CIs.

**Stratified PPI++** fixes all three problems: it respects regional structure *and* leverages the global proxy signal with per-region bias correction.

In [ ]:
# Option A: LLM judge on all conversations
judge_estimate = ClassicalMeanEstimator().estimate(dataset=dataset, y_field="y_proxy")

# Option B: Human labels only, all regions pooled
human_estimate = ClassicalMeanEstimator().estimate(dataset=dataset, y_field="y_true")

# Option C: Per-region human estimates (averaged)
per_region_estimates = {}
for region_id, region_name in region_names.items():
    region_data = [r for r in labeled if r.get("stratum_id") == region_id]
    region_dataset = labeled.__class__(region_data)
    per_region_estimates[region_id] = ClassicalMeanEstimator().estimate(dataset=region_dataset, y_field="y_true")

# Compute global estimate from per-region estimates (population-proportional weights)
total_labeled = sum(len([r for r in labeled if r.get("stratum_id") == region_id]) for region_id in region_names)
pooled_mean = sum(
    (len([r for r in labeled if r.get("stratum_id") == region_id]) / total_labeled)
    * per_region_estimates[region_id].mean
    for region_id in region_names
)
pooled_std = np.sqrt(
    sum(
        (len([r for r in labeled if r.get("stratum_id") == region_id]) / total_labeled) ** 2
        * per_region_estimates[region_id].std ** 2
        for region_id in region_names
    )
)
z_alpha = 1.96  # 95% CI
pooled_lower = pooled_mean - z_alpha * pooled_std
pooled_upper = pooled_mean + z_alpha * pooled_std

# Extract CI bounds for cleaner formatting
judge_lo = judge_estimate.confidence_interval.lower_bound
judge_hi = judge_estimate.confidence_interval.upper_bound
human_lo = human_estimate.confidence_interval.lower_bound
human_hi = human_estimate.confidence_interval.upper_bound

sep = "-" * 78
print(sep)
print(f"{'Method':<48} {'Estimate':>8}   {'95% CI':>18}")
print(sep)
print(f"{'A. LLM Judge (all regions)':<48} {judge_estimate.mean:>7.1%}   [{judge_lo:.1%}, {judge_hi:.1%}]")
print(f"{'B. Human labels only (pooled)':<48} {human_estimate.mean:>7.1%}   [{human_lo:.1%}, {human_hi:.1%}]")
print(f"{'C. Per-region human, then averaged':<48} {pooled_mean:>7.1%}   [{pooled_lower:.1%}, {pooled_upper:.1%}]")
print(sep)

## The root cause: the LLM judge is biased everywhere, worse in Asia

Per-region analysis shows the problem clearly: the judge underestimates hallucinations in all regions, but the bias is especially severe in Asia.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

region_list = [0, 1, 2]
x_pos = np.arange(len(region_list))
bar_width = 0.35
colors_list = [C_NA, C_EU, C_ASIA]
region_labels = ["North America", "Europe", "Asia"]

judge_means = []
human_means = []
biases = []

for region_id in region_list:
    region_labeled = [r for r in labeled if r.get("stratum_id") == region_id]
    region_dataset = labeled.__class__(region_labeled)

    proxy_on_labeled = region_dataset["y_proxy"]
    true_on_labeled = region_dataset["y_true"]

    judge_mean = np.mean(proxy_on_labeled)
    human_mean = np.mean(true_on_labeled)
    bias = judge_mean - human_mean

    judge_means.append(judge_mean)
    human_means.append(human_mean)
    biases.append(bias)

# Plot bars
ax.bar(
    x_pos - bar_width / 2,
    judge_means,
    bar_width,
    label="LLM Judge (on annotated subset)",
    color=C_JUDGE,
    edgecolor="white",
    linewidth=2,
    zorder=3,
)
ax.bar(
    x_pos + bar_width / 2,
    human_means,
    bar_width,
    label="Human Annotation (ground truth)",
    color=C_HUMAN,
    edgecolor="white",
    linewidth=2,
    zorder=3,
)

# Annotate biases
for i, (j_mean, h_mean, bias) in enumerate(zip(judge_means, human_means, biases)):
    max_y = max(j_mean, h_mean)
    ax.annotate(
        "",
        xy=(i + bar_width / 2 + 0.05, h_mean + 0.01),
        xytext=(i - bar_width / 2 - 0.05, j_mean + 0.01),
        arrowprops=dict(arrowstyle="<->", color="#666666", lw=2),
    )
    ax.text(
        i,
        max_y + 0.04,
        f"Bias={bias:+.1%}",
        ha="center",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#FFFDE7", edgecolor="#CCCCCC"),
    )

ax.set_xticks(x_pos)
ax.set_xticklabels(region_labels, fontsize=11)
ax.set_ylabel("Hallucination Rate", fontsize=12)
ax.set_title("LLM Judge Bias Varies by Region (Worst in Asia)", fontsize=13, fontweight="bold")
ax.set_ylim(0, 0.20)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="upper left", fontsize=10)
plt.tight_layout()
plt.show()

print("\nPer-region bias (Judge - Human):")
for region_id, region_label in region_names.items():
    print(f"  {region_label:<15} : {biases[region_id]:+.1%}")

## Stratified PPI++ adapts per region

`StratifiedPPIMeanEstimator` implements Stratified PPI++:

1. **Split** the dataset by region
2. **Compute per-region bias correction** using the labeled subset in each region
3. **Tune lambda independently** for each region (per-region signal-to-noise ratio)
4. **Combine** per-region estimates with population-proportional weights

This way, Asia's worse judge quality doesn't drag down the overall estimate — each region gets an optimal correction tailored to its own signal characteristics.

In [ ]:
estimator = StratifiedPPIMeanEstimator()

stratified_result = estimator.estimate(
    dataset,
    y_true_field="y_true",
    y_proxy_field="y_proxy",
    groups_field="stratum_id",
    metric_name="Global Hallucination Rate",
    confidence_level=0.95,
    power_tuning=True,
)

print(stratified_result.summary())

### Exporting the Result as JSON

The `InferenceResult` object serializes directly to a plain dictionary, ready for logging, dashboards, or downstream pipelines.

In [ ]:
print(to_json(stratified_result))

## Stratified PPI++ vs Standard PPI: Respecting Data Structure Wins

The plot below compares all estimators:

- **LLM judge**: narrow CI but wrong (biased)
- **Human-only**: unbiased but very wide
- **Standard PPI++** (ignores regions): pools all data, uses single lambda — misses the regional structure
- **Stratified PPI++** (our method): unbiased *and* narrow — adapts lambda per region to account for differing judge quality

In [ ]:
# Standard PPI++ on pooled data (ignores stratum_id)
pooled_dataset = labeled.__class__(
    [{k: v for k, v in r.items() if k != "stratum_id" and k != "region"} for r in dataset]
)

standard_ppi_result = PPIMeanEstimator().estimate(
    pooled_dataset,
    y_true_field="y_true",
    y_proxy_field="y_proxy",
    metric_name="Global Hallucination Rate",
    confidence_level=0.95,
)

# Compute true global mean (for reference)
true_global_mean = np.mean([r["y_true"] for r in labeled if "y_true" in r])

TRUE_RATE = true_global_mean

estimates = [
    (
        f"LLM Judge\n(N={len(dataset):,} | raw proxy)",
        judge_estimate.mean,
        judge_estimate.confidence_interval.lower_bound,
        judge_estimate.confidence_interval.upper_bound,
        C_JUDGE,
    ),
    (
        f"Human Annotation\n(n={len(labeled):,} | small sample)",
        human_estimate.mean,
        human_estimate.confidence_interval.lower_bound,
        human_estimate.confidence_interval.upper_bound,
        C_HUMAN,
    ),
    (
        f"Standard PPI++\n(n={stratified_result.n_true} + N={stratified_result.n_proxy}\nignores regions)",
        standard_ppi_result.mean,
        standard_ppi_result.confidence_interval.lower_bound,
        standard_ppi_result.confidence_interval.upper_bound,
        "#95A5A6",
    ),
    (
        f"Stratified PPI++ (GLIDE)\n(n={stratified_result.n_true} + N={stratified_result.n_proxy}\nper-region lambda)",
        stratified_result.mean,
        stratified_result.confidence_interval.lower_bound,
        stratified_result.confidence_interval.upper_bound,
        C_GLIDE,
    ),
]

fig, ax = plt.subplots(figsize=(12, 6))
y_pos = [3, 2, 1, 0]

for y, (label, mean, lo, hi, color) in zip(y_pos, estimates):
    # Confidence interval line
    ax.plot([lo, hi], [y, y], color=color, linewidth=5, solid_capstyle="round", zorder=3)
    # Cap marks
    for xc in [lo, hi]:
        ax.plot([xc, xc], [y - 0.2, y + 0.2], color=color, linewidth=3, zorder=3)
    # Point estimate
    ax.scatter(mean, y, s=250, color=color, zorder=5, edgecolors="white", linewidths=2.5)
    # Value label above
    ax.text(mean, y + 0.38, f"{mean:.1%}", ha="center", va="bottom", fontsize=12, color=color, fontweight="bold")
    # Confidence interval bounds below
    ax.text(mean, y - 0.38, f"[{lo:.1%}, {hi:.1%}]", ha="center", va="top", fontsize=10, color="#888888")

# True value
ax.axvline(TRUE_RATE, color=C_TRUTH, linestyle="--", linewidth=2.5, zorder=4)
ax.text(TRUE_RATE + 0.004, 3.75, f"True rate {TRUE_RATE:.1%}", color=C_TRUTH, fontsize=11, fontweight="bold")

ax.set_yticks(y_pos)
ax.set_yticklabels([e[0] for e in estimates], fontsize=10.5)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlabel("Estimated Global Hallucination Rate", fontsize=12)
title = "Stratified PPI++ Respects Regional Structure & Achieves Narrower CIs"
ax.set_title(title, fontsize=14, fontweight="bold")
ax.set_xlim(-0.01, 0.22)
ax.set_ylim(-0.8, 4.2)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(left=False)
plt.tight_layout()
plt.show()

## Why Stratified PPI++ Has a Narrower Confidence Interval

When strata have **different proxy quality**, standard PPI++ uses a single global lambda that balances the signal-to-noise across all regions. Stratified PPI++ computes an optimal lambda **per region**, allowing:

- **North America & Europe** (good judge correlation): high lambda → leverage more judge signal
- **Asia** (poor judge correlation): lower lambda → trust less in judge, rely more on labeled data

This adaptive tuning reduces the total variance compared to using one global lambda.

In [ ]:
# Extract CI widths
judge_width = judge_estimate.confidence_interval.upper_bound - judge_estimate.confidence_interval.lower_bound
human_width = human_estimate.confidence_interval.upper_bound - human_estimate.confidence_interval.lower_bound
standard_ppi_width = (
    standard_ppi_result.confidence_interval.upper_bound - standard_ppi_result.confidence_interval.lower_bound
)
stratified_width = stratified_result.confidence_interval.upper_bound - stratified_result.confidence_interval.lower_bound

fig, ax = plt.subplots(figsize=(9, 5))

methods = ["LLM Judge", "Human-only", "Standard PPI++", "Stratified PPI++"]
widths = [judge_width, human_width, standard_ppi_width, stratified_width]
colors = [C_JUDGE, C_HUMAN, "#95A5A6", C_GLIDE]

bars = ax.bar(methods, widths, color=colors, edgecolor="white", linewidth=2, zorder=3)

# Annotate bars
for bar, width in zip(bars, widths):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        height,
        f"{width:.1%}",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
    )

# Highlight the winner
ax.text(3, stratified_width + 0.015, "← Narrowest!", fontsize=11, color=C_GLIDE, fontweight="bold")

ax.set_ylabel("95% Confidence Interval Width", fontsize=12)
ax.set_title("Stratified PPI++ Achieves Tighter Confidence Intervals", fontsize=13, fontweight="bold")
ax.set_ylim(0, max(widths) * 1.15)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## Testing hypothesis: Is the true rate significantly different from 8%?

Stratified PPI++ enables formal hypothesis testing with precise confidence intervals. Let's test if the global hallucination rate is significantly different from North America's observed judge estimate (8%).

In [ ]:
z_stat, p_value, _ = stratified_result.confidence_interval.test_null_hypothesis(
    h0_value=0.08,  # Judge's estimate for NA
    alternative="two-sided",
)

sep = "-" * 60
print("Hypothesis Test — Stratified PPI++ Estimator")
print(sep)
print("H0 : global hallucination rate = 8%   (Judge's NA estimate)")
print("H1 : global hallucination rate ≠ 8%   (Different from NA)")
print()
print(f"z-statistic : {z_stat:.2f}")
print(f"p-value     : {p_value:.6f}")
print()
if p_value < 0.05:
    print("Decision  : Reject H0 at the 5% level.")
    print("=> The global hallucination rate is significantly different from 8%.")
    print("=> This makes sense: Asia's 15% rate pulls the global average up.")
else:
    print("Decision  : Cannot reject H0 at the 5% level.")

## Summary: Stratified PPI++ adapts to regional data structure

| | Judge | Human-only | Standard PPI++ | **Stratified PPI++** |
|---|-------|-----------|---------|----------|
| Sample size | 4,000 | 300 | 4,300 | 4,300 |
| Unbiased | ❌ | ✅ | ✅ | ✅ |
| Respects regional structure | ❌ | ⚠️ | ❌ | ✅ |
| Narrow CI | 🟠 *(misleading)* | ❌ | ✅ | ✅✅ |
| Labeling cost | Low | **High** | Small | Small |

**Key takeaways:**

1. **Regional bias matters.** A global LLM judge may have systematically different error rates across regions. Ignoring this structure leaves precision on the table.

2. **Per-region lambda is powerful.** Stratified PPI++ automatically adapts the bias correction per region, unlocking tighter confidence intervals than standard PPI++ when strata differ in proxy quality.

3. **Stratified PPI++ is a natural extension.** It uses the same PPI++ machinery per region, then combines with population-proportional weights. No new assumptions — just respecting the data structure.

4. **Regional breakdowns are actionable.** You can request the per-region estimates from the same estimator, enabling regional monitoring dashboards and interventions.

---

*Want to go further? Stratified PPI++ applies wherever your data has natural partitions: by language, data source, time period, customer segment, product variant, or any other meaningful grouping. The tighter the correlation within strata and the more heterogeneous the strata, the larger your precision gain.*